In [1]:
import pandas as pd
import re


In [2]:
df = pd.read_csv("../data/raw/workout_sessions_messy.csv")

In [3]:
df

,user_id,date,exercise,sets,reps,weight_kg,session_duration_hr,calories_burned,avg_bpm,max_bpm,water_intake_l,workout_type,workout_difficulty
0,U1186,2024-01-25,cycling,NaN,NaN,NaN,NaN,198.0,156.0,176,1.07,Cardio,7
1,U0316,2024-01-16,Bench Press,3.0,5.0,NaN,0.56,235.0,90.0,112,0.73,STRENGTH,Level 4
2,U0614,2024-08-03,Cycling,NaN,NaN,NaN,1.28,438.0,149.0,163,NaN,Cardio,10
3,U0355,2024-10-26,Squat,4.0,5.0,NaN,0.51,237.0,100.0,120,0.49,Strength,9.0
4,U0780,2024-07-02,Rowing,NaN,NaN,NaN,0.83,235.0,153.0,177,0.52,Cardio,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
153745,U0896,2024-08-17,Running,NaN,NaN,NaN,0.62,425.0,129.0,147,0.32,Cardio,8-10
153746,U1043,2024-02-05,Squat,3.0,5.0,83.3,1.06,415.0,133.0,153,0.58,Strength,10 out of 10
153747,U0499,2024-10-06,Cycling,NaN,NaN,NaN,1.18,300.0,152.0,166,0.65,Cardio,NaN
153748,U0103,2024-11-14,Rowing,NaN,NaN,NaN,1.49,544.0,111.0,134,0.66,Cardio,3-10


In [4]:
df.set_index("user_id", inplace=True)

In [5]:
df.index.duplicated().sum()

151353

# Exercise

In [6]:
df["exercise"].unique()

array(['cycling', 'Bench Press', 'Cycling', 'Squat', 'Rowing', 'HIIT',
       ' Bench Press ', 'Running', 'Deadlift', 'Yoga', 'ROW', 'deadlift',
       'yoga', ' Rowing ', ' Deadlift ', 'Jogging', 'running', ' HIIT ',
       'High Intensity Interval Training', 'YOGA', 'hiit', 'BENCH PRESS',
       'Bike', 'Dead Lift', 'H.I.I.T', 'CYCLING', ' RUN ', ' Running ',
       'Indoor Rowing', 'BenchPress', 'squat', 'bench press', ' Yoga ',
       ' deadlift ', 'Yoga Flow', 'DEADLIFT', 'RUN', ' Cycling ',
       ' Squat ', ' rowing ', 'SQUAT', ' Indoor Rowing ', 'rowing',
       'Back Squat', ' running ', ' ROW ', ' DEADLIFT ', ' yoga ',
       ' CYCLING ', ' YOGA ', ' BenchPress ', ' bench press ',
       ' Back Squat ', ' SQUAT ', ' Yoga Flow ', ' Dead Lift ',
       ' cycling ', ' hiit ', ' BENCH PRESS ', ' Jogging ', ' Bike ',
       ' H.I.I.T ', ' High Intensity Interval Training ', ' squat '],
      dtype=object)

In [7]:
df["exercise"] = df["exercise"].str.strip()

In [8]:
df["exercise"].unique()

array(['cycling', 'Bench Press', 'Cycling', 'Squat', 'Rowing', 'HIIT',
       'Running', 'Deadlift', 'Yoga', 'ROW', 'deadlift', 'yoga',
       'Jogging', 'running', 'High Intensity Interval Training', 'YOGA',
       'hiit', 'BENCH PRESS', 'Bike', 'Dead Lift', 'H.I.I.T', 'CYCLING',
       'RUN', 'Indoor Rowing', 'BenchPress', 'squat', 'bench press',
       'Yoga Flow', 'DEADLIFT', 'rowing', 'SQUAT', 'Back Squat'],
      dtype=object)

In [9]:
# lower amount of options so now will reduce by finding out common values

df["exercise"] = (
    df["exercise"].str.upper().replace({
        "BENCH PRESS": "BENCHPRESS",
        "HIGH INTENSITY INTERVAL TRAINING": "HIIT",
        "H.I.I.T": "HIIT",
        "DEAD LIFT": "DEADLIFT",
        "BIKE": "CYCLING",
        "YOGA FLOW": "YOGA",
        "ROWING": "ROW",
        "BACK SQUAT": "SQUAT",
        "INDOOR ROWING": "ROW"
    })
)

In [10]:
df["exercise"].unique()

array(['CYCLING', 'BENCHPRESS', 'SQUAT', 'ROW', 'HIIT', 'RUNNING',
       'DEADLIFT', 'YOGA', 'JOGGING', 'RUN'], dtype=object)

# Sets

In [11]:
df[["exercise", "sets"]]

,exercise,sets
user_id,,
U1186,CYCLING,NaN
U0316,BENCHPRESS,3.0
U0614,CYCLING,NaN
U0355,SQUAT,4.0
U0780,ROW,NaN
...,...,...
U0896,RUNNING,NaN
U1043,SQUAT,3.0
U0499,CYCLING,NaN


In [12]:
df.groupby("exercise")["sets"].apply(lambda x: x.notna().sum())

exercise
BENCHPRESS    18437
CYCLING           0
DEADLIFT      18212
HIIT              0
JOGGING           0
ROW               0
RUN               0
RUNNING           0
SQUAT         18440
YOGA              0
Name: sets, dtype: int64

In [13]:
df["workout_category"] = df["exercise"].apply(
    lambda x: "WEIGHT" if x in ["BENCHPRESS", "DEADLIFT", "SQUAT"] else "NW" # NW stands for no weight
)

In [14]:
strength_exercise = ["BENCHPRESS", "DEADLIFT", "SQUAT"]

df.loc[~df["exercise"].isin(strength_exercise), "reps"] = None


## Weight_kg depending on the workout

In [15]:
df.loc[df["workout_type"] == "CARDIO", "weight_kg"] = None

In [16]:
df[df["workout_type"] == "STRENGTH"]["weight_kg"].isna().sum()

120

In [52]:
df["sets"].isna().sum()

98661

In [17]:
# did any users exercise more than once
df.reset_index()[["user_id", "date"]].duplicated().sum()

22328

In [18]:
df

,date,exercise,sets,reps,weight_kg,session_duration_hr,calories_burned,avg_bpm,max_bpm,water_intake_l,workout_type,workout_difficulty,workout_category
user_id,,,,,,,,,,,,,
U1186,2024-01-25,CYCLING,NaN,NaN,NaN,NaN,198.0,156.0,176,1.07,Cardio,7,NW
U0316,2024-01-16,BENCHPRESS,3.0,5.0,NaN,0.56,235.0,90.0,112,0.73,STRENGTH,Level 4,WEIGHT
U0614,2024-08-03,CYCLING,NaN,NaN,NaN,1.28,438.0,149.0,163,NaN,Cardio,10,NW
U0355,2024-10-26,SQUAT,4.0,5.0,NaN,0.51,237.0,100.0,120,0.49,Strength,9.0,WEIGHT
U0780,2024-07-02,ROW,NaN,NaN,NaN,0.83,235.0,153.0,177,0.52,Cardio,NaN,NW
...,...,...,...,...,...,...,...,...,...,...,...,...,...
U0896,2024-08-17,RUNNING,NaN,NaN,NaN,0.62,425.0,129.0,147,0.32,Cardio,8-10,NW
U1043,2024-02-05,SQUAT,3.0,5.0,83.3,1.06,415.0,133.0,153,0.58,Strength,10 out of 10,WEIGHT
U0499,2024-10-06,CYCLING,NaN,NaN,NaN,1.18,300.0,152.0,166,0.65,Cardio,NaN,NW


# Workout Type

In [19]:
df["workout_type"].unique()

array(['Cardio', 'STRENGTH', 'Strength', 'CARDIO', ' HIIT ', 'HIIT',
       'Mobility', 'strength', ' Mobility ', 'HIIT ', 'MOBILITY', 'Str',
       ' Strength ', ' Str ', ' Cardio ', 'hiit', 'Endurance', 'cardio',
       'mobility', 'High Intensity', 'Flexibility', ' mobility ',
       ' HIIT  ', ' CARDIO ', ' strength ', ' STRENGTH ', ' cardio ',
       ' Endurance ', ' hiit ', ' Flexibility ', ' MOBILITY ',
       ' High Intensity '], dtype=object)

In [20]:
df['workout_type'].isna().sum()

0

In [21]:
df["workout_type"] = df["workout_type"].str.strip()

In [22]:
df["workout_type"].unique()

array(['Cardio', 'STRENGTH', 'Strength', 'CARDIO', 'HIIT', 'Mobility',
       'strength', 'MOBILITY', 'Str', 'hiit', 'Endurance', 'cardio',
       'mobility', 'High Intensity', 'Flexibility'], dtype=object)

In [23]:
df["workout_type"] = (
    df["workout_type"].str.upper().replace(
        {"Cardio": "CARDIO",
         "STRENGTH": "STRENGTH",
         "STRENGTH": "STRENGTH",
         "STR": "STRENGTH",
         "Mobility": "MOBILITY",
         "mobility": "MOBILITY",
         "HIGH INTENSITY": "HIIT",
         "Endurance": "ENDURANCE",
         "Flexibility": "FLEXIBILITY"
         }
    )
)

In [24]:
df["workout_type"].unique()

array(['CARDIO', 'STRENGTH', 'HIIT', 'MOBILITY', 'ENDURANCE',
       'FLEXIBILITY'], dtype=object)

## Workout difficulty

In [25]:
df["workout_difficulty"].unique()

array(['7 ', 'Level 4', '10 ', '9.0', nan, '5 (easy)', '4.0', '7', '4-10',
       '4 ', '6', '4/10', '3', '5 ', '7 (easy)', 'ten', '8 ', ' 5', ' 3',
       '9 out of 10', ' 6', '1 (easy)', ' 4', 'five', '2', '2 ', '2-10',
       '5 out of 10', '9/10', 'Level 7', 'Level 6', '4', '10/10',
       '4 (easy)', '2/10', '6 ', '8', 'Level 8', ' 8', '9 (hard)', '5.0',
       '9-10', ' 1', '6/10', 'Level 3', '9', '1 ', '3.0', '2.0', '8/10',
       '7 out of 10', '7/10', '7-10', '3 out of 10', '8 out of 10', '1',
       '10 out of 10', '2 out of 10', '8-10', '4 out of 10', '10 (hard)',
       '1-10', '5/10', '6 (easy)', '5', '3 ', ' 9', 'Level 9', '2 (easy)',
       '1.0', '1/10', '10-10', '6-10', 'Level 5', '5-10', 'Level 10',
       ' 2', '3-10', 'Level 2', '10', ' 7', '7.0', '6 out of 10',
       '1 out of 10', '6.0', '8.0', '8 (hard)', '10.0', '3 (easy)',
       '3/10', '9 ', 'Level 1', ' 10'], dtype=object)

In [26]:
df["workout_difficulty"] = df["workout_difficulty"].str.strip()

In [27]:
df["workout_difficulty"].unique()

array(['7', 'Level 4', '10', '9.0', nan, '5 (easy)', '4.0', '4-10', '4',
       '6', '4/10', '3', '5', '7 (easy)', 'ten', '8', '9 out of 10',
       '1 (easy)', 'five', '2', '2-10', '5 out of 10', '9/10', 'Level 7',
       'Level 6', '10/10', '4 (easy)', '2/10', 'Level 8', '9 (hard)',
       '5.0', '9-10', '1', '6/10', 'Level 3', '9', '3.0', '2.0', '8/10',
       '7 out of 10', '7/10', '7-10', '3 out of 10', '8 out of 10',
       '10 out of 10', '2 out of 10', '8-10', '4 out of 10', '10 (hard)',
       '1-10', '5/10', '6 (easy)', 'Level 9', '2 (easy)', '1.0', '1/10',
       '10-10', '6-10', 'Level 5', '5-10', 'Level 10', '3-10', 'Level 2',
       '7.0', '6 out of 10', '1 out of 10', '6.0', '8.0', '8 (hard)',
       '10.0', '3 (easy)', '3/10', 'Level 1'], dtype=object)

In [28]:
def extract_difficulty(value):
    if pd.isna(value):
        return None
    
    value = str(value).lower().strip()
    numbers = re.findall(r'\d+\.?\d*', value)
    
    if len(numbers) == 0:
        return None
    
    num = float(numbers[0])   # take first number from the list
    
    if 1 <= num <= 10:
        return num
    else:
        return None

df["workout_difficulty"] = df["workout_difficulty"].apply(extract_difficulty)

In [29]:
df["workout_difficulty"].mean().round()

5.0

In [30]:
df["workout_difficulty"].isna().sum()

37975

too many na's refill with the average

In [31]:
df["workout_difficulty"] = df["workout_difficulty"].fillna(df["workout_difficulty"].mean().round())

In [32]:
df["workout_difficulty"].isna().sum()

0

In [33]:
df

,date,exercise,sets,reps,weight_kg,session_duration_hr,calories_burned,avg_bpm,max_bpm,water_intake_l,workout_type,workout_difficulty,workout_category
user_id,,,,,,,,,,,,,
U1186,2024-01-25,CYCLING,NaN,NaN,NaN,NaN,198.0,156.0,176,1.07,CARDIO,7.0,NW
U0316,2024-01-16,BENCHPRESS,3.0,5.0,NaN,0.56,235.0,90.0,112,0.73,STRENGTH,4.0,WEIGHT
U0614,2024-08-03,CYCLING,NaN,NaN,NaN,1.28,438.0,149.0,163,NaN,CARDIO,10.0,NW
U0355,2024-10-26,SQUAT,4.0,5.0,NaN,0.51,237.0,100.0,120,0.49,STRENGTH,9.0,WEIGHT
U0780,2024-07-02,ROW,NaN,NaN,NaN,0.83,235.0,153.0,177,0.52,CARDIO,5.0,NW
...,...,...,...,...,...,...,...,...,...,...,...,...,...
U0896,2024-08-17,RUNNING,NaN,NaN,NaN,0.62,425.0,129.0,147,0.32,CARDIO,8.0,NW
U1043,2024-02-05,SQUAT,3.0,5.0,83.3,1.06,415.0,133.0,153,0.58,STRENGTH,10.0,WEIGHT
U0499,2024-10-06,CYCLING,NaN,NaN,NaN,1.18,300.0,152.0,166,0.65,CARDIO,5.0,NW


# Session duration hr

In [34]:
df['session_duration_hr'].min()

0.3

In [35]:
df['session_duration_hr'].max()

7.0

In [36]:
df["session_duration_hr"].isna().sum()

6097

In [37]:
df["session_duration_hr"] = df["session_duration_hr"].fillna(df["session_duration_hr"].mean().round())

In [38]:
df["session_duration_hr"].isna().sum()

0

In [39]:
df

,date,exercise,sets,reps,weight_kg,session_duration_hr,calories_burned,avg_bpm,max_bpm,water_intake_l,workout_type,workout_difficulty,workout_category
user_id,,,,,,,,,,,,,
U1186,2024-01-25,CYCLING,NaN,NaN,NaN,1.00,198.0,156.0,176,1.07,CARDIO,7.0,NW
U0316,2024-01-16,BENCHPRESS,3.0,5.0,NaN,0.56,235.0,90.0,112,0.73,STRENGTH,4.0,WEIGHT
U0614,2024-08-03,CYCLING,NaN,NaN,NaN,1.28,438.0,149.0,163,NaN,CARDIO,10.0,NW
U0355,2024-10-26,SQUAT,4.0,5.0,NaN,0.51,237.0,100.0,120,0.49,STRENGTH,9.0,WEIGHT
U0780,2024-07-02,ROW,NaN,NaN,NaN,0.83,235.0,153.0,177,0.52,CARDIO,5.0,NW
...,...,...,...,...,...,...,...,...,...,...,...,...,...
U0896,2024-08-17,RUNNING,NaN,NaN,NaN,0.62,425.0,129.0,147,0.32,CARDIO,8.0,NW
U1043,2024-02-05,SQUAT,3.0,5.0,83.3,1.06,415.0,133.0,153,0.58,STRENGTH,10.0,WEIGHT
U0499,2024-10-06,CYCLING,NaN,NaN,NaN,1.18,300.0,152.0,166,0.65,CARDIO,5.0,NW


## Calories Burned

In [40]:
df.drop(columns="calories_burned")

,date,exercise,sets,reps,weight_kg,session_duration_hr,avg_bpm,max_bpm,water_intake_l,workout_type,workout_difficulty,workout_category
user_id,,,,,,,,,,,,
U1186,2024-01-25,CYCLING,NaN,NaN,NaN,1.00,156.0,176,1.07,CARDIO,7.0,NW
U0316,2024-01-16,BENCHPRESS,3.0,5.0,NaN,0.56,90.0,112,0.73,STRENGTH,4.0,WEIGHT
U0614,2024-08-03,CYCLING,NaN,NaN,NaN,1.28,149.0,163,NaN,CARDIO,10.0,NW
U0355,2024-10-26,SQUAT,4.0,5.0,NaN,0.51,100.0,120,0.49,STRENGTH,9.0,WEIGHT
U0780,2024-07-02,ROW,NaN,NaN,NaN,0.83,153.0,177,0.52,CARDIO,5.0,NW
...,...,...,...,...,...,...,...,...,...,...,...,...
U0896,2024-08-17,RUNNING,NaN,NaN,NaN,0.62,129.0,147,0.32,CARDIO,8.0,NW
U1043,2024-02-05,SQUAT,3.0,5.0,83.3,1.06,133.0,153,0.58,STRENGTH,10.0,WEIGHT
U0499,2024-10-06,CYCLING,NaN,NaN,NaN,1.18,152.0,166,0.65,CARDIO,5.0,NW


In [41]:
df

,date,exercise,sets,reps,weight_kg,session_duration_hr,calories_burned,avg_bpm,max_bpm,water_intake_l,workout_type,workout_difficulty,workout_category
user_id,,,,,,,,,,,,,
U1186,2024-01-25,CYCLING,NaN,NaN,NaN,1.00,198.0,156.0,176,1.07,CARDIO,7.0,NW
U0316,2024-01-16,BENCHPRESS,3.0,5.0,NaN,0.56,235.0,90.0,112,0.73,STRENGTH,4.0,WEIGHT
U0614,2024-08-03,CYCLING,NaN,NaN,NaN,1.28,438.0,149.0,163,NaN,CARDIO,10.0,NW
U0355,2024-10-26,SQUAT,4.0,5.0,NaN,0.51,237.0,100.0,120,0.49,STRENGTH,9.0,WEIGHT
U0780,2024-07-02,ROW,NaN,NaN,NaN,0.83,235.0,153.0,177,0.52,CARDIO,5.0,NW
...,...,...,...,...,...,...,...,...,...,...,...,...,...
U0896,2024-08-17,RUNNING,NaN,NaN,NaN,0.62,425.0,129.0,147,0.32,CARDIO,8.0,NW
U1043,2024-02-05,SQUAT,3.0,5.0,83.3,1.06,415.0,133.0,153,0.58,STRENGTH,10.0,WEIGHT
U0499,2024-10-06,CYCLING,NaN,NaN,NaN,1.18,300.0,152.0,166,0.65,CARDIO,5.0,NW


## Session Duration

In [42]:
df['session_duration_hr'].min()

0.3

In [43]:
df['session_duration_hr'].max()

7.0

In [44]:
df['session_duration_hr'].mean()

0.9405823739837399

In [45]:
df.groupby(["user_id", "date"]).size().gt(1).any()

True

In [46]:
df.groupby(["user_id", "date"])["session_duration_hr"].sum()

user_id  date      
 U0001   2024-02-12    1.14
         2024-02-24    0.67
         2024-03-04    0.30
         2024-03-09    0.76
         2024-06-18    0.45
                       ... 
U1200    2024-12-20    0.47
         2024-12-26    0.63
         2024-12-29    0.42
         2024/07/04    0.75
         30/04/2024    0.84
Name: session_duration_hr, Length: 131422, dtype: float64

In [47]:
df.groupby("user_id")["session_duration_hr"].sum()

user_id
 U0001       6.23
 U0002       6.31
 U0003       5.17
 U0004       1.85
 U0005       9.88
            ...  
U1196       92.61
U1197      131.39
U1198       96.89
U1199      102.72
U1200      133.88
Name: session_duration_hr, Length: 2397, dtype: float64

In [48]:
df["session_duration_hr"].mean()

0.9405823739837399

In [49]:
df['session_duration_hr'].isna().sum()

0

In [50]:
df.to_csv("../data/processed/workout_session.csv")